# Emotion Contagion ABM

This notebook is a light front end for the refactored simulation project.

Use it when you want a notebook workflow, but keep the core engine in the Python modules.

In [ ]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from build_simulation import run_multiple_simulations, save_results
from metrics import load_many_conditions, leader_score_table, summary_table, temporal_variance_table
from run_simulation import read_config, config_from_dict

sns.set(style="whitegrid")

## 1. Load a YAML config and run one condition or a small batch

In [ ]:
config_path = Path("default.yaml")
raw = read_config(config_path)
config = config_from_dict(raw)

# Example edits
config.total_population = 15
config.runs = 3
config.leader_style = "Free"
config.network.structure = "community"
config.network.include_leader_intimacy = True
config.rl.enabled = True
config.output.base_dir = "simulation_runs"
config.output.verbose = True

results = run_multiple_simulations(config)
output_folder = save_results(results, config)
print(f"Saved to: {output_folder}")

## 2. Quick summary of the runs you just saved

In [ ]:
summary = pd.DataFrame(
    {
        "Run": [run.run_id for run in results.runs],
        "Final_Avg_Emotion": [np.mean(run.emotion_history[-1]) for run in results.runs],
        "Final_SD": [np.std(run.emotion_history[-1]) for run in results.runs],
        "Interventions": [len(run.intervention_log) for run in results.runs],
    }
)
summary

## 3. Load multiple saved conditions for comparison

In [ ]:
date_str = config.output.date_folder or date.today().strftime("%m_%d_%Y")

results_dict = load_many_conditions(
    base_dir=config.output.base_dir,
    date_str=date_str,
    team_size=config.total_population,
    network_structures=["community", "random", "core_periphery"],
    styles=[
        "No_Intervention",
        "High_Fully_Constrained",
        "Low_Fully_Constrained",
        "High_Initially_Constrained",
        "Low_Initially_Constrained",
        "Free",
    ],
)

In [ ]:
leader_score_table(results_dict).head()

In [ ]:
summary_table(results_dict).head()

## 4. Example plot: temporal variance

In [ ]:
df_temp = temporal_variance_table(results_dict)

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_temp, x="Time", y="Variance", hue="Leader", errorbar="sd")
plt.title("Temporal Variance of Emotions by Leader")
plt.show()